# 02 — Perfilamento inicial e decisão de modelagem

O perfilamento examina as 20 colunas dos três arquivos de origem, a unicidade das chaves, a multiplicidade de itens por pedido, ausências, domínios e cobertura das junções. Esses resultados fundamentam a escolha de uma fato com uma linha por pedido e dimensões de data e localidade.

Perguntas de negócio: (1) Como varia a taxa de atraso das entregas por estado do cliente? (2) Como evolui a taxa de atraso por mês da compra? (3) Como a taxa de atraso se distribui entre faixas de frete total por pedido?


## Contrato das métricas

A unidade analítica é um pedido, identificado por order_id. A taxa tem como denominador apenas pedidos com order_status igual a delivered e datas real e estimada de entrega presentes. O numerador são os elegíveis cuja data de calendário real é posterior à estimada. A hora é ignorada. Estado é o do cliente associado ao pedido; mês é o da compra. O frete total soma as linhas de itens por pedido; a comparação é descritiva e não causal. Grupos vazios não têm taxa definida.


In [ ]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("catalogo", "")
catalogo = dbutils.widgets.get("catalogo").strip() or spark.sql("SELECT current_catalog()").first()[0]
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo), "Informe um catálogo válido no widget."
assert catalogo != "hive_metastore", "Use o catálogo do workspace com Unity Catalog."
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
origem = f"/Volumes/{catalogo}/olist_mvp_raw/originais"
silver = f"{catalogo}.olist_mvp_silver"
gold = f"{catalogo}.olist_mvp_gold"
arquivos = {
    "pedidos": "olist_orders_dataset.csv",
    "itens": "olist_order_items_dataset.csv",
    "clientes": "olist_customers_dataset.csv",
}
colunas_esperadas = {
    "pedidos": ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
                "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"],
    "itens": ["order_id", "order_item_id", "product_id", "seller_id",
              "shipping_limit_date", "price", "freight_value"],
    "clientes": ["customer_id", "customer_unique_id", "customer_zip_code_prefix",
                 "customer_city", "customer_state"],
}
print(f"Catálogo: {catalogo}; volume: {origem}")


In [ ]:
def ler_origem(nome):
    caminho = f"{origem}/{arquivos[nome]}"
    quadro = (spark.read.option("header", "true")
              .option("inferSchema", "false")
              .option("sep", ",")
              .option("encoding", "UTF-8")
              .option("mode", "FAILFAST")
              .csv(caminho))
    assert quadro.columns == colunas_esperadas[nome], (
        f"Esquema inesperado em {arquivos[nome]}: {quadro.columns}"
    )
    return quadro


In [ ]:
quadros = {nome: ler_origem(nome) for nome in arquivos}
for nome, quadro in quadros.items():
    print(f"{nome}: {quadro.count()} linhas")
    quadro.printSchema()
    display(quadro.limit(5))


## Completude, domínios e faixas por atributo

Nulo e texto vazio são ausências. Mínimo e máximo textuais ajudam a detectar formatos, mas não representam percentis numéricos. Nenhuma linha é removida nesta etapa.


In [ ]:
perfil = []
for nome, quadro in quadros.items():
    total = quadro.count()
    expressoes = [
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in quadro.columns
    ]
    ausencias = quadro.agg(*expressoes).first().asDict()
    for coluna in quadro.columns:
        extremos = quadro.agg(F.min(coluna).alias("min"), F.max(coluna).alias("max")).first()
        perfil.append((nome, coluna, total, int(ausencias[coluna] or 0),
                       round(100.0 * (ausencias[coluna] or 0) / total, 2) if total else None,
                       str(extremos["min"]) if extremos["min"] is not None else None,
                       str(extremos["max"]) if extremos["max"] is not None else None))
display(spark.createDataFrame(
    perfil, ["arquivo", "coluna", "linhas", "ausentes", "ausencia_pct", "min_textual", "max_textual"]
))


## Unicidade, cardinalidade e cobertura das junções


In [ ]:
def problemas_de_chave(quadro, colunas, nome):
    nulos = quadro.filter(F.expr(" OR ".join(f"{c} IS NULL OR trim({c}) = ''" for c in colunas))).count()
    grupos = quadro.groupBy(*colunas).count().filter("count > 1")
    return (nome, quadro.count(), nulos, grupos.count(),
            grupos.agg(F.sum("count").alias("n")).first()["n"] or 0)

chaves = [
    problemas_de_chave(quadros["pedidos"], ["order_id"], "pedidos.order_id"),
    problemas_de_chave(quadros["itens"], ["order_id", "order_item_id"], "itens.(order_id, order_item_id)"),
    problemas_de_chave(quadros["clientes"], ["customer_id"], "clientes.customer_id"),
]
display(spark.createDataFrame(chaves, ["chave", "linhas", "chaves_nulas", "grupos_duplicados", "linhas_em_grupos_duplicados"]))
print("Máximo de itens por pedido:",
      quadros["itens"].groupBy("order_id").count().agg(F.max("count")).first()[0])


In [ ]:
pedidos = quadros["pedidos"]
itens = quadros["itens"]
clientes = quadros["clientes"]
cobertura = [
    ("pedidos sem cliente", pedidos.join(clientes, "customer_id", "left_anti").count()),
    ("pedidos sem itens", pedidos.join(itens.select("order_id").distinct(), "order_id", "left_anti").count()),
    ("itens sem pedido", itens.join(pedidos.select("order_id"), "order_id", "left_anti").count()),
]
display(spark.createDataFrame(cobertura, ["verificacao", "registros"]))


## Consistência temporal e numérica

A ausência da data real em pedidos não entregues é esperada; em pedidos delivered ela pede investigação. Valores negativos, erros de conversão e sequências de datas incoerentes são contados, mas não excluídos automaticamente. Acurácia externa não pode ser provada por esta base; a verificação aqui é de plausibilidade interna.


In [ ]:
status = pedidos.groupBy("order_status").agg(
    F.count("*").alias("pedidos"),
    F.sum(F.when(F.col("order_delivered_customer_date").isNull() |
                 (F.trim("order_delivered_customer_date") == ""), 1).otherwise(0)).alias("sem_entrega_real"),
    F.sum(F.when(F.col("order_estimated_delivery_date").isNull() |
                 (F.trim("order_estimated_delivery_date") == ""), 1).otherwise(0)).alias("sem_previsao")
)
display(status.orderBy("order_status"))

datas = pedidos.select(
    "order_id", "order_status",
    *[F.expr(f"try_cast({c} AS TIMESTAMP)").alias(c) for c in
      ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
       "order_delivered_customer_date", "order_estimated_delivery_date"]]
)
display(datas.agg(
    F.sum(F.when(F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"), 1).otherwise(0)).alias("entrega_antes_compra"),
    F.sum(F.when(F.col("order_estimated_delivery_date") < F.col("order_purchase_timestamp"), 1).otherwise(0)).alias("previsao_antes_compra"),
    F.sum(F.when((F.col("order_status") == "delivered") & F.col("order_delivered_customer_date").isNull(), 1).otherwise(0)).alias("entregue_sem_data_real")
))


In [ ]:
itens_tipados = itens.select(
    F.expr("try_cast(price AS DECIMAL(18,2))").alias("preco"),
    F.expr("try_cast(freight_value AS DECIMAL(18,2))").alias("frete")
)
display(itens_tipados.agg(
    F.min("preco").alias("preco_min"), F.max("preco").alias("preco_max"),
    F.min("frete").alias("frete_min"), F.max("frete").alias("frete_max"),
    F.sum(F.when(F.col("preco") < 0, 1).otherwise(0)).alias("preco_negativo"),
    F.sum(F.when(F.col("frete") < 0, 1).otherwise(0)).alias("frete_negativo"),
    F.expr("percentile(preco, array(0.25, 0.5, 0.75, 0.99))").alias("percentis_preco"),
    F.expr("percentile(frete, array(0.25, 0.5, 0.75, 0.99))").alias("percentis_frete")
))


## Validação de formatos e extremos por atributo

As regras de formato abaixo são testes de plausibilidade, não uma verificação externa de endereço ou pagamento. Valores acima de Q3 + 1,5 × IQR são sinalizados para inspeção, sem remoção automática.


In [ ]:
verificacoes = []
def registrar(nome, coluna, regra, condicao):
    verificacoes.append((nome, coluna, regra, quadros[nome].filter(condicao).count()))

for nome, colunas in {
    "pedidos": ["order_id", "customer_id"],
    "itens": ["order_id", "product_id", "seller_id"],
    "clientes": ["customer_id", "customer_unique_id"],
}.items():
    for coluna in colunas:
        registrar(nome, coluna, "identificador fora de 32 caracteres hexadecimais",
                  F.col(coluna).isNotNull() & ~F.col(coluna).rlike("^[0-9a-fA-F]{32}$"))

for nome, colunas in {
    "pedidos": ["order_purchase_timestamp", "order_approved_at",
                "order_delivered_carrier_date", "order_delivered_customer_date"],
    "itens": ["shipping_limit_date"],
}.items():
    for coluna in colunas:
        registrar(nome, coluna, "data e hora não conversíveis",
                  F.col(coluna).isNotNull() & (F.trim(F.col(coluna)) != "") &
                  F.expr(f"try_cast({coluna} AS TIMESTAMP)").isNull())

registrar("pedidos", "order_estimated_delivery_date", "data não conversível",
          F.col("order_estimated_delivery_date").isNotNull() &
          (F.trim("order_estimated_delivery_date") != "") &
          F.expr("try_cast(order_estimated_delivery_date AS DATE)").isNull())
registrar("itens", "order_item_id", "sequência não positiva ou não numérica",
          F.expr("try_cast(order_item_id AS INT)").isNull() |
          (F.expr("try_cast(order_item_id AS INT)") <= 0))
for coluna in ["price", "freight_value"]:
    registrar("itens", coluna, "valor negativo ou não conversível",
              (F.col(coluna).isNotNull() & (F.trim(F.col(coluna)) != "")) &
              (F.expr(f"try_cast({coluna} AS DECIMAL(18,2))").isNull() |
               (F.expr(f"try_cast({coluna} AS DECIMAL(18,2))") < 0)))
registrar("clientes", "customer_zip_code_prefix", "CEP prefixo fora de 1 a 5 dígitos",
          F.col("customer_zip_code_prefix").isNotNull() &
          ~F.col("customer_zip_code_prefix").rlike("^[0-9]{1,5}$"))
registrar("clientes", "customer_city", "cidade vazia",
          F.col("customer_city").isNull() | (F.trim("customer_city") == ""))
registrar("clientes", "customer_state", "UF fora de duas letras",
          F.col("customer_state").isNotNull() &
          ~F.col("customer_state").rlike("^[A-Za-z]{2}$"))
display(spark.createDataFrame(verificacoes, ["arquivo", "coluna", "regra", "violacoes"]))

for coluna in ["price", "freight_value"]:
    q1, q3 = itens_tipados.agg(
        F.expr(f"percentile({'preco' if coluna == 'price' else 'frete'}, array(0.25, 0.75))")
    ).first()[0]
    if q1 is not None and q3 is not None:
        limite = float(q3) + 1.5 * (float(q3) - float(q1))
        acima = itens_tipados.filter(F.col("preco" if coluna == "price" else "frete") > limite).count()
        print(f"{coluna}: limite exploratório IQR superior = {limite:.2f}; acima = {acima}. Nenhuma exclusão automática.")
